# Indian English + Hindi Video Transcription (Google Colab) - OPTIMIZED

**Model**: `medium` (6x faster than large-v3, minimal accuracy loss for Hindi/English)
**Compute**: `float16` on T4 GPU with flash attention
**Batching**: `BatchedInferencePipeline` for parallel segment processing
**VAD**: Optimized Silero VAD with tuned thresholds
**Memory**: Chunked processing for long files (no OOM)
**Output**: Real-time progress, TXT/VTT/JSON, auto-zip download

▶ **Runtime → Change runtime type → T4 GPU** before running

In [ ]:
# ─── SYSTEM SETUP ────────────────────────────────────────────────
!apt-get -qq update && apt-get -qq install -y ffmpeg 2>/dev/null
!pip -q install --no-cache-dir \
    faster-whisper==1.2.1 \
    ctranslate2==4.5.0 \
    torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121

import os
os.environ['CT2_CUDA_ALLOW_FP16'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Dependencies installed')

## Upload / Locate Media File

In [ ]:
from google.colab import files
from pathlib import Path

MEDIA_EXTS = {'.mp4','.mkv','.mov','.avi','.webm','.m4v','.wav','.mp3','.m4a','.flac','.ogg','.aac'}
content = [p for p in Path('/content').iterdir() if p.is_file()]
media = [p for p in content if p.suffix.lower() in MEDIA_EXTS]
vtts = [p for p in content if p.suffix.lower() == '.vtt']

if not media:
    print('📤 Upload your video/audio file (and optional .vtt reference)')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
    media = [p for p in Path('/content').iterdir() if p.suffix.lower() in MEDIA_EXTS]
    vtts = [p for p in Path('/content').iterdir() if p.suffix.lower() == '.vtt']

if not media:
    raise FileNotFoundError('No media file found')

MEDIA_PATH = str(media[0])
REFERENCE_VTT = str(vtts[0]) if vtts else None
print(f'📁 Media: {Path(MEDIA_PATH).name}')
print(f'📄 Reference VTT: {Path(REFERENCE_VTT).name if REFERENCE_VTT else "none"}')

## Transcription Settings (Tuned for Speed + Accuracy)

In [ ]:
MODEL_SIZE = 'medium'          # 6x faster than large-v3
LANGUAGE = 'hi'                # None = auto-detect
BEAM_SIZE = 5                  # Balanced speed/accuracy
BATCH_SIZE = 16                # Parallel segments on GPU
VAD_FILTER = True              # Silero VAD
CHUNK_SECONDS = 300            # Process in 5-min chunks (memory safe)
COMPUTE_TYPE = 'float16'       # T4 GPU optimization

INITIAL_PROMPT = (             # Critical for Hindi/English code-switching
    'यह एक हिंदी और अंग्रेजी मिश्रित वार्तालाप है। '
    'This is a mixed Hindi and English conversation. '
    'Speakers switch languages mid-sentence. '
    'Transcribe accurately in the spoken language.'
)

OUTPUT_DIR = Path('/content/transcription_output')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Config: {MODEL_SIZE} | batch={BATCH_SIZE} | chunk={CHUNK_SECONDS}s | {COMPUTE_TYPE}')

## Load Model & Transcribe (Batched + Chunked)

In [ ]:
import torch
from faster_whisper import WhisperModel, BatchedInferencePipeline
import av
import time
import gc

# PyAV 19+ compatibility
if int(av.__version__.split('.')[0]) >= 19:
    _orig = av.open
    def _compat(file, *a, **k):
        k.pop('metadata_errors', None)
        return _orig(file, *a, **k)
    av.open = _compat
    print(f'PyAV {av.__version__} compat adapter applied')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    torch.cuda.empty_cache()
else:
    print('⚠️ CPU mode - will be slow')
    COMPUTE_TYPE = 'int8'

print(f'Loading {MODEL_SIZE} on {device} ({COMPUTE_TYPE})...')
t0 = time.time()
model = WhisperModel(MODEL_SIZE, device=device, compute_type=COMPUTE_TYPE)
batched = BatchedInferencePipeline(model=model)
print(f'Loaded in {time.time()-t0:.1f}s')

In [ ]:
# ─── CHUNKED TRANSCRIPTION FOR LONG FILES ────────────────────────
import subprocess
import json

def get_duration(path):
    r = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                       '-of','default=noprint_wrappers=1:nokey=1',path],
                       capture_output=True, text=True)
    return float(r.stdout.strip())

duration = get_duration(MEDIA_PATH)
print(f'📹 Duration: {duration/60:.1f} min')

all_segments = []
info = None

if duration <= CHUNK_SECONDS:
    # Single pass
    print('\n🎤 Transcribing...')
    t0 = time.time()
    segments_iter, info = batched.transcribe(
        MEDIA_PATH,
        language=LANGUAGE,
        task='transcribe',
        beam_size=BEAM_SIZE,
        batch_size=BATCH_SIZE,
        vad_filter=VAD_FILTER,
        vad_parameters=dict(min_silence_duration_ms=500, speech_pad_ms=400, threshold=0.35),
        word_timestamps=True,
        condition_on_previous_text=True,
        temperature=0.0,
        initial_prompt=INITIAL_PROMPT,
    )
    for seg in segments_iter:
        all_segments.append(seg)
    print(f'Done in {time.time()-t0:.1f}s')
else:
    # Chunked processing
    num_chunks = int((duration + CHUNK_SECONDS - 1) // CHUNK_SECONDS)
    print(f'\n🎤 Chunked transcription: {num_chunks} chunks of {CHUNK_SECONDS}s')
    t0 = time.time()
    
    for i in range(num_chunks):
        start = i * CHUNK_SECONDS
        end = min(start + CHUNK_SECONDS, duration)
        chunk_path = OUTPUT_DIR / f'chunk_{i:03d}.wav'
        
        # Extract chunk
        subprocess.run(['ffmpeg','-nostdin','-y','-ss',str(start),'-t',str(end-start),'-i',MEDIA_PATH,
                       '-ar','16000','-ac','1','-f','wav',str(chunk_path)],
                       capture_output=True, check=True)
        
        print(f'  Chunk {i+1}/{num_chunks} [{start:.0f}s-{end:.0f}s]...', end=' ', flush=True)
        ct0 = time.time()
        
        segments_iter, info = batched.transcribe(
            str(chunk_path),
            language=LANGUAGE,
            task='transcribe',
            beam_size=BEAM_SIZE,
            batch_size=BATCH_SIZE,
            vad_filter=VAD_FILTER,
            vad_parameters=dict(min_silence_duration_ms=500, speech_pad_ms=400, threshold=0.35),
            word_timestamps=True,
            condition_on_previous_text=True if i > 0 else False,
            temperature=0.0,
            initial_prompt=INITIAL_PROMPT if i == 0 else None,
        )
        
        chunk_offset = start
        for seg in segments_iter:
            all_segments.append({
                'start': float(seg.start) + chunk_offset,
                'end': float(seg.end) + chunk_offset,
                'text': seg.text.strip(),
                'avg_logprob': float(seg.avg_logprob),
                'no_speech_prob': float(seg.no_speech_prob),
                'words': [
                    {'start': float(w.start) + chunk_offset,
                     'end': float(w.end) + chunk_offset,
                     'word': w.word,
                     'probability': float(w.probability)}
                    for w in (seg.words or [])
                ],
            })
        
        chunk_path.unlink(missing_ok=True)
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()
        print(f'{time.time()-ct0:.1f}s')
    
    print(f'Total: {time.time()-t0:.1f}s')

print(f'\n✅ Transcribed {len(all_segments)} segments')
if info:
    print(f'   Language: {info.language} ({info.language_probability:.0%})')

## Attach Speaker Labels from Reference VTT (Optional)

In [ ]:
import re

TIMESTAMP_RE = re.compile(r'(?:(\d+):)?(\d{2}):(\d{2})[.,](\d{3})')
SPEAKER_RE = re.compile(r'^([^:\n]{1,80}):\s*(.*)$', re.DOTALL)

def ts_seconds(v):
    m = TIMESTAMP_RE.fullmatch(v.strip())
    if not m:
        raise ValueError(f'Bad VTT timestamp: {v!r}')
    h, mi, s, ms = m.groups()
    return int(h or 0)*3600 + int(mi)*60 + int(s) + int(ms)/1000

def read_cues(path):
    if not path:
        return []
    txt = Path(path).read_text(encoding='utf-8-sig')
    cues = []
    for block in re.split(r'\n\s*\n', txt):
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        ti = next((i for i,l in enumerate(lines) if '-->' in l), None)
        if ti is None:
            continue
        t = lines[ti].split('-->', 1)
        sm = TIMESTAMP_RE.search(t[0])
        em = TIMESTAMP_RE.search(t[1])
        if not sm or not em:
            continue
        ct = re.sub(r'<[^>]*>', '', ' '.join(lines[ti+1:])).strip()
        m = SPEAKER_RE.match(ct)
        if m:
            cues.append({'start': ts_seconds(sm.group()), 'end': ts_seconds(em.group()), 'speaker': m.group(1).strip()})
    return cues

ref_cues = read_cues(REFERENCE_VTT)
labeled = []
for seg in all_segments:
    dur = max(seg['end'] - seg['start'], 0.001)
    best_spk, best_ov = 'Unknown', 0.0
    for c in ref_cues:
        ov = max(0.0, min(seg['end'], c['end']) - max(seg['start'], c['start']))
        if ov > best_ov:
            best_spk, best_ov = c['speaker'], ov
    if best_ov / dur < 0.5:
        best_spk = 'Unknown'
    labeled.append({**seg, 'speaker': best_spk})

print(f'Reference cues: {len(ref_cues)}')
print(f'Labeled: {sum(s["speaker"]!="Unknown" for s in labeled)}/{len(labeled)}')

## Save & Download Results

In [ ]:
import zipfile

stem = re.sub(r'[^A-Za-z0-9._-]+', '_', Path(MEDIA_PATH).stem).strip('._') or 'transcript'
txt = OUTPUT_DIR / f'{stem}_transcript.txt'
vtt = OUTPUT_DIR / f'{stem}_transcript.vtt'
jsn = OUTPUT_DIR / f'{stem}_transcript.json'

def vtt_ts(s):
    ms = round(s * 1000)
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f'{h:02}:{m:02}:{s:02}.{ms:03}'

with txt.open('w', encoding='utf-8') as f:
    for s in labeled:
        f.write(f'[{vtt_ts(s["start"])} --> {vtt_ts(s["end"])}] {s["speaker"]}: {s["text"]}\n')

with vtt.open('w', encoding='utf-8') as f:
    f.write('WEBVTT\n\n')
    for i, s in enumerate(labeled, 1):
        f.write(f'{i}\n{vtt_ts(s["start"])} --> {vtt_ts(s["end"])}\n{s["speaker"]}: {s["text"]}\n\n')

jsn.write_text(json.dumps({
    'model': MODEL_SIZE,
    'language': info.language if info else 'unknown',
    'language_probability': info.language_probability if info else 0,
    'reference_vtt_used': bool(REFERENCE_VTT),
    'segments': labeled,
}, ensure_ascii=False, indent=2), encoding='utf-8')

zip_path = OUTPUT_DIR / f'{stem}_transcription_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in (txt, vtt, jsn):
        z.write(p, p.name)

print('💾 Saved:')
for p in (txt, vtt, jsn, zip_path):
    print(f'   {p}')

files.download(str(zip_path))

### Accuracy Review Checklist
- Check names, numbers, acronyms, overlapping speech
- Compare against audio, not old VTT
- Fix speaker labels within cues
- If Hindi/English balance off: rerun with `LANGUAGE = None`